### Guardrails with Langchain:
 - What are Guardrails & Why do they matter?
 - Two approaches: Deterministic vs Model-based
 -  Built-in: PII Detection Middleware
 - Built-in: Human-in-the-Loop Middleware
 - Custom: Before-Agent Guardrail (input filtering)
 - Custom: After-Agent Guardrail (output safety)
 - Layered / Combined Guardrails
 -  Real-World Use Case: Healthcare Chatbot

In [1]:
#Installation:
from dotenv import load_dotenv
load_dotenv()

#Setup API keys:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

### What are Guardrails?
Guardrails help you build safe, compliant AI applications by validating and filtering content at key points in your agent's execution.

They are implemented as middleware that intercepts execution:

 - Before the agent starts (input guardrails)
 - After it completes (output guardrails)
 - Around model and tool calls





# Two Approaches to Guardrails:

### Deterministic Guardrails:
 - Rule-based: regex, keyword matching, explicit checks
 - ✅ Fast, predictable, cost-effective
 - ❌ May miss nuanced violations

### Model-Based Guardrails:
 - Uses LLMs/classifiers for semantic understanding
 - ✅ Catches subtle/nuanced issues
 - ❌ Slower and more expensive

## Common Use Cases

| Use Case | Example |
|----------|---------|
| PII leakage prevention | Redact emails/credit cards before logging |
| Prompt injection blocking | Detect adversarial inputs |
| Harmful content filtering | Block dangerous requests |
| Business rule enforcement | Require approval for financial ops |
| Output quality validation | Ensure response meets safety standards |

In [8]:
### Quick Illustration of two approaches to Guardrails:
import re

# 1. Deterministic Guardrails: These are rules that are hard-coded into the system. They are designed to prevent the AI from taking certain actions or making certain decisions. For example, a deterministic guardrail might prevent the AI from accessing certain sensitive data or from making decisions that could harm users.

def deterministic_guardrail(input_text:str)-> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit","malware", "bomb", "attack"]
    return any(kw in input_text.lower() for kw in banned_keywords)

test_inputs = [
    "How to hack into a computer system?",
    "What are the best practices for cybersecurity?",
    "explain how to create malware",
]
print("=== Deterministic Guardrail Results ===")
for inp in test_inputs:
    if deterministic_guardrail(inp):
        print(f"Blocked: {inp}")
    else:
        print(f"Allowed: {inp}")

=== Deterministic Guardrail Results ===
Blocked: How to hack into a computer system?
Allowed: What are the best practices for cybersecurity?
Blocked: explain how to create malware


In [ ]:
#Model based approach:
from langchain_openai import ChatOpenAI

def model_based_guardrail(input_text:str)-> str:
    """use a LLM to evaluate content safefty. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
    prompt = f""" Is the following user input safe or unsafe? Please respond with only 'SAFE' or 'UNSAFE'.

    User Input: {input_text}

    """
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()

print("=== Model-based Guardrail Results ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if verdict == "UNSAFE" else "SAFE"
    print(f"Input: {inp} | Verdict: {status}")

 

### Built-in Guardrail — PII Detection Middleware:
 - LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

#Define a simple dummy tool:
def customer_lookup(query:str)-> str:
    """Dummy tool to simulate a customer lookup."""
    return f"Customer info for query: {query}"

#create agent with PII middleware:
agent = create_agent(
    model = "gpt-4o-mini",
    tools = [customer_lookup],
    middleware = [
        # Redact emails in user input before sending to the model
        PIIMiddleware(
            "email",  # Type of PII to redact
            strategy="redact",  # Redaction strategy
            apply_to_input=True,  # Apply to user input
        ),
        #Mask credit cards in user input before sending to the model
        PIIMiddleware(
            "credit_card",  # Type of PII to mask
            strategy="mask",  # Masking strategy
            apply_to_input=True,  # Apply to user input
        ),

        #block API keys in user input before sending to the model---raise error if detected
        PIIMiddleware(
            "api_key",  # Type of PII to block
            detector = r"sk-[A-Za-z0-9]{48}",  # Regex pattern to detect API keys
            strategy="block",  # Blocking strategy
            apply_to_input=True,  # Apply to user input
        ),
    ],
)

print("=== Agent with PII Middleware created successfully ===")

In [ ]:
#Test PII Redaction:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is test@example.com and my credit card number is 4111 1111 1111 1111. Can you help?"
    }]
})

print("Agent Response after PII Redaction:")
print(result["messages"][-1].content)


In [ ]:
result

In [ ]:
#Test API Key Blocking:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "My OpenAI API key is sk-1234567890abcdef1234567890abcdef1234567890abcdef. Can you help?"
        }]
    })
except Exception as e:
    print("Agent Response after API Key Blocking:")
    print(e)

### Built-in Guardrail — Human-in-the-Loop Middleware:
 - Pauses agent execution before sensitive operations and waits for human approval.

## Best for:
 - Financial transactions
 - Sending emails to external parties
 - Deleting production data
 - Any operation with significant business impact
 
Key requirement: A checkpointer for state persistence across interrupts.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for query: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to the specified recipient."""
    return f"Email sent to {to} with subject '{subject}' and body '{body}'"

@tool
def delete_record(table: str, condition: str) -> str:
    """Delete a record from the database."""
    return f"Deleted records from  {table} where {condition}"

#Create an agent with Human-in-the-Loop middleware:
hitl_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_web, send_email, delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # Require human approval for sending emails
                "delete_record": True,  # Require human approval for deleting records
                "search_web": False,  # No human approval needed for web searches
            }
        )
    ],
    checkpointer = InMemorySaver() #required for state persistence across human approvals
)
print("=== Agent with Human-in-the-Loop Middleware created successfully ===")

In [ ]:
# Step 1: Invoke — agent will pause before send_email:
config = {"configurable":{"thread_id":"session_001"}}

result = hitl_agent.invoke({
    "messages": [{"role": "user", "content": "Please send an email to shamser@company.com about quarterly report."}],
    "config": config
})

print("=== Agent paused --- awaiting human approval ===")
print(result["messages"][-1].content)  # This will show the agent's response indicating it's awaiting approval

In [ ]:
#step 2 : Human review and approval:
# Simulate human approval by sending a follow-up message to the agent
approved_result = hitl_agent.invoke({
    Command(resume={"decision": [{"type": "approve"}]})"}),
    config = config #same thread_id to continue the same session
)

print("=== Approved! final response ===")
print(approved_result["messages"][-1].content)  # This will show the agent's final response after approval

In [ ]:
#step 3 : Human review and rejection:
config2 = {"configurable":{"thread_id":"session_002"}}

hitl_agent.invoke(
    "messages": [{"role": "user", "content": "Please delete the record from the users table where active= false."}],
    "config": config2
)


rejected_result = hitl_agent.invoke(
    Command(resume={"decision": [{"type": "reject", "reason": "This action is not allowed."}]}),
    config = config2 #same thread_id to continue the same session
)

print("=== Rejected! final response ===")
print(rejected_result["messages"][-1].content)  # This will show the agent

### Custom Guardrail — Before-Agent Hook (Input Filter):

Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

   - Keyword/content filtering
   - Authentication checks
   - Rate limiting
   -  Blocking specific categories of requests_